<a href="https://colab.research.google.com/github/hwangho-kim/Transformer_Fewshot_PdM/blob/main/Megpie_RUL_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from scipy.optimize import curve_fit
import datetime

# 한글 폰트 설정 (Windows: 'Malgun Gothic', Mac: 'AppleGothic')
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False # 마이너스 기호 깨짐 방지

# ==========================================
# 0. 가상 데이터(Dummy Data) 생성
# ==========================================
def generate_dummy_data():
    """웨이브, 노이즈, 이상치가 포함된 비선형 열화 데이터를 생성합니다."""
    np.random.seed(42)
    times = pd.date_range(start='2026-05-01', end='2026-05-30', freq='1h') # 1시간 단위
    n = len(times)

    # 기본 열화 베이스라인 (처음엔 0 유지하다가 중간 이후 지수적 증가)
    onset_idx = int(n * 0.5)
    base_val = np.zeros(n)
    base_val[onset_idx:] = np.exp(np.linspace(0, 3.5, n - onset_idx)) / np.exp(3.5) * 1.3

    # 노이즈 및 웨이브(사인파) 추가
    noise = np.random.normal(0, 0.03, n)
    wave = np.sin(np.linspace(0, 30, n)) * 0.08
    median_val = base_val + noise + wave

    # 이상치(Outlier) 추가
    median_val[int(n * 0.2)] = 0.9  # 튀는 값
    median_val[int(n * 0.7)] = 0.2  # 갑자기 떨어지는 값

    # 0 이하의 값은 0으로 클리핑 (보통 센서값이 음수가 되지 않는다고 가정)
    median_val = np.clip(median_val, 0, None)

    return pd.DataFrame({'end_time': times, 'median_val': median_val})

sel_df = generate_dummy_data()

# ==========================================
# 1. 데이터 전처리 (Preprocessing)
# ==========================================
df = sel_df.set_index('end_time').copy()

# 1-1. 이상치 완화를 위한 Moving Median (Window=5)
df['median_clean'] = df['median_val'].rolling(window=5, center=True).median()
df['median_clean'] = df['median_clean'].bfill().ffill() # 결측치 처리

# 1-2. Butterworth Low Pass Filter (노이즈 및 잔물결 제거)
def apply_lowpass_filter(data, cutoff_freq, fs, order=4):
    nyq = 0.5 * fs
    normal_cutoff = cutoff_freq / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    y = filtfilt(b, a, data)
    return y

# 샘플링 주파수 fs=1(1시간에 1번), Cutoff: 24시간 주기의 변동 제거 (1/24)
df['smoothed'] = apply_lowpass_filter(df['median_clean'], cutoff_freq=1/24, fs=1)

# 1-3. 6시간 단위 리샘플링 (안전을 위해 최대값 Max 사용)
df_6h = df.resample('6h').max()
df_6h = df_6h.dropna()

# ==========================================
# 2. 열화 시작점 탐지 (Degradation Onset)
# ==========================================
# 방법: 스무딩된 값이 임계값(0.15)을 연속 3번 초과하는 시점 찾기
threshold = 0.15
consecutive_points = 3

is_above = df_6h['smoothed'] > threshold
onset_time = None
count = 0

for idx, val in is_above.items():
    if val:
        count += 1
        if count >= consecutive_points:
            # 최초로 초과하기 시작한 시점을 onset으로 정의
            onset_time = idx - pd.Timedelta(hours=6*(consecutive_points-1))
            break
    else:
        count = 0

# ==========================================
# 3. 모델 피팅 및 RUL 예측 (Extrapolation)
# ==========================================
failure_limit = 1.5
current_time = df_6h.index[-1]
predicted_failure_time = None

if onset_time is None:
    print("열화 시작점이 탐지되지 않았습니다. (정상 상태)")
else:
    # Onset 이후의 데이터만 추출하여 학습에 사용
    fit_data = df_6h[df_6h.index >= onset_time].copy()

    # 시간 데이터를 수치형(시간 단위)으로 변환 (fitting 안정성을 위함)
    x_data = (fit_data.index - onset_time).total_seconds() / 3600.0
    y_data = fit_data['smoothed'].values

    # 지수 함수 정의: y = a * exp(b * x) + c
    def exp_func(x, a, b, c):
        # 오버플로우 방지를 위해 제한
        return a * np.exp(np.clip(b * x, -700, 700)) + c

    try:
        # 단조 증가를 보장하기 위해 bounds 설정 (a > 0, b > 0)
        popt, _ = curve_fit(exp_func, x_data, y_data, bounds=([0, 0, -np.inf], [np.inf, np.inf, np.inf]), maxfev=10000)
        a, b, c = popt

        # 예측: a * exp(b * t) + c = 1.5 가 되는 t를 계산
        if failure_limit > c:
            t_failure_hours = np.log((failure_limit - c) / a) / b
            predicted_failure_time = onset_time + pd.Timedelta(hours=t_failure_hours)
            rul = predicted_failure_time - current_time

            print(f"--- RUL 예측 결과 ---")
            print(f"현재 시점(Current): {current_time.strftime('%Y-%m-%d %H:%M')}")
            print(f"열화 시작(Onset):   {onset_time.strftime('%Y-%m-%d %H:%M')}")
            print(f"예상 고장 시점:     {predicted_failure_time.strftime('%Y-%m-%d %H:%M')}")
            print(f"남은 수명(RUL):     {rul.total_seconds() / 3600:.1f} 시간 ({rul.days}일 {rul.components.hours}시간)")
        else:
            print("수식 구조상 1.5에 도달할 수 없습니다.")

    except Exception as e:
        print(f"곡선 적합(Curve Fitting) 실패: {e}")

# ==========================================
# 4. 결과 시각화 (Visualization)
# ==========================================
plt.figure(figsize=(14, 7))

# 원본 및 전처리 데이터 플롯
plt.plot(df.index, df['median_val'], color='lightgray', label='Original Sensor Data (1H)', alpha=0.6)
plt.plot(df_6h.index, df_6h['smoothed'], color='blue', marker='o', markersize=4, label='Smoothed & Resampled Data (6H Max)')

# Upper Limit 선
plt.axhline(y=failure_limit, color='red', linestyle='--', linewidth=2, label=f'Upper Limit ({failure_limit})')

if onset_time is not None:
    # 열화 시작점 표시
    plt.axvline(x=onset_time, color='orange', linestyle=':', linewidth=2, label='Degradation Onset')
    plt.plot(onset_time, df_6h.loc[onset_time, 'smoothed'], 'go', markersize=8, label='Onset Point')

    if predicted_failure_time is not None:
        # Extrapolation (예측 곡선) 그리기
        # 시작점부터 예상 고장 시점 + 약간의 여유 공간까지 시간축 생성
        future_times = pd.date_range(start=onset_time, end=predicted_failure_time + pd.Timedelta(days=2), freq='6h')
        future_x = (future_times - onset_time).total_seconds() / 3600.0
        future_y = exp_func(future_x, *popt)

        # 현재 시간 이후의 곡선만 점선으로 그리기 위해 필터링
        pred_mask = future_times >= current_time
        plt.plot(future_times[pred_mask], future_y[pred_mask], color='red', linestyle='-', linewidth=2, label='Predicted Trend (Extrapolation)')

        # 피팅에 사용된 구간은 실선으로 표시 (선택사항)
        fit_mask = future_times <= current_time
        plt.plot(future_times[fit_mask], future_y[fit_mask], color='green', linestyle='-', linewidth=2, label='Fitted Trend')

        # 고장 시점 마커
        plt.plot(predicted_failure_time, failure_limit, 'rX', markersize=10, label='Predicted Failure Point')

# 현재 시점 선
plt.axvline(x=current_time, color='black', linestyle='--', linewidth=1.5, label='Current Time')

plt.title("Megpie 부품 수명(RUL) 예측 (Low Pass Filter & Curve Fitting)", fontsize=16)
plt.xlabel("시간 (Time)", fontsize=12)
plt.ylabel("센서 값 (Sensor Value)", fontsize=12)
plt.ylim(0, 1.8)
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()